# Reproducing the analysis: cross-population transfer and recalibration of ciprofloxacin resistance prediction in *E. coli*

This notebook runs top to bottom and reproduces the numbers in the manuscript (Tables 2-10, the Norway and South Asia
checks, and the South Asia BioProject sensitivity check) from one file in this repository:
`data/processed/qc_cohort_final_features.csv` (7,274 quality-controlled genomes, eight engineered features, phenotype,
country, collection year, BioProject and MLST).

- It does not rebuild the features from the raw BV-BRC downloads; that step is in `notebooks/corrected_pipeline.py`.
- Nothing below is typed in by hand: every number printed is computed in the cell that prints it.
- **Run in Google Colab:** File > Open notebook > GitHub tab > paste `Rita127-neup/ecoli-amr-calibration` > open this file.
  Run the first cell, then **Runtime > Restart session** if Colab asks, then use **Runtime > Run all**.
  Run time is a few minutes.

In [1]:
# Pin the library versions used for the manuscript (Python 3.13, Google Colab).
%pip install -q scikit-learn==1.6.1 pandas==2.2.3 numpy==2.1.3

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, warnings
import numpy as np, pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression as LR
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, matthews_corrcoef,
                             confusion_matrix, brier_score_loss)
from sklearn.exceptions import ConvergenceWarning

# newer SciPy versions warn that scikit-learn 1.6.1 passes an unused solver option; the warning does not affect results
warnings.filterwarnings('ignore', message='Unknown solver options')

print('scikit-learn', sklearn.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)
if sklearn.__version__ != '1.6.1':
    print('WARNING: the manuscript used scikit-learn 1.6.1; other versions can change the third decimal.')

# Random seed for the saga solver of the elastic-net model. The saga solver stops at one of two nearby solutions that
# differ only in the weakly identified qnrB coefficient (0.707 or 0.712); which one depends on the seed. The manuscript's
# first run had no seed and landed on 0.712. Seed 4 reproduces that run, and with it every number in the manuscript.
# Any other seed changes qnrB by at most 0.006 and nothing else beyond the third decimal.
SEED = 4

FEATURES = ['gyrA_fq_mut', 'parC_fq_mut', 'qnrA', 'qnrB', 'qnrS', 'oqxA', 'oqxB', 'aac6_Ib_cr']
LOCAL = '../data/processed/qc_cohort_final_features.csv'
URL = ('https://raw.githubusercontent.com/Rita127-neup/ecoli-amr-calibration/main/'
       'data/processed/qc_cohort_final_features.csv')
d = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL, dtype={'Genome ID': str})
d['y'] = (d['Resistant Phenotype'] == 'Resistant').astype(int)
d['Isolation Country'] = d['Isolation Country'].replace({'England': 'United Kingdom'})
print(d.shape, '| data read from:', LOCAL if os.path.exists(LOCAL) else URL)

scikit-learn 1.6.1 | pandas 2.2.3 | numpy 2.1.3
(7274, 20) | data read from: ../data/processed/qc_cohort_final_features.csv


## 1. Cohort and populations (Tables 1-2)

In [3]:
uk = d[d['Isolation Country'] == 'United Kingdom']
uk_model = uk.dropna(subset=['MLST'])          # the one UK genome without an MLST assignment is excluded from modelling
norway = d[d['Isolation Country'] == 'Norway']
sasia = d[d['Isolation Country'].isin(['India', 'Pakistan', 'Bangladesh'])]
pops = {'UK': uk_model, 'Norway': norway, 'South Asia': sasia}

assert len(d) == 7274 and len(uk) == 2575 and len(uk_model) == 2574 and len(norway) == 3153 and len(sasia) == 238, 'cohort changed'
print(f"quality-controlled cohort: {len(d)} genomes, {d['y'].sum()} resistant, {(d['y'] == 0).sum()} susceptible")
print(f"with a resolvable isolation country: {d['Isolation Country'].notna().sum()}")

ST131 = 'MLST.ecoli_achtman_4.131'
rows = []
for name, df in [('UK (all, 2575)', uk), ('Norway', norway), ('South Asia', sasia)]:
    rows.append({'population': name, 'N': len(df), 'resistant': int(df['y'].sum()),
                 'resistant %': round(100 * df['y'].mean(), 1),
                 'years': f"{int(df['Collection Year'].min())}-{int(df['Collection Year'].max())}",
                 'missing BioProject %': round(100 * df['BioProject Accession'].isna().mean(), 1),
                 'year missing': int(df['Collection Year'].isna().sum())})
print(pd.DataFrame(rows).to_string(index=False))
print('\nUK year missing among the 2,574 modelled genomes:', int(uk_model['Collection Year'].isna().sum()))
for r, name in [(d[d['Isolation Country'].isin(['India', 'Pakistan', 'Bangladesh'])], 'South Asia')]:
    print(name, 'by country (resistant/susceptible):',
          {c: f"{int((g['y'] == 1).sum())}R/{int((g['y'] == 0).sum())}S" for c, g in r.groupby('Isolation Country')})

quality-controlled cohort: 7274 genomes, 1843 resistant, 5431 susceptible
with a resolvable isolation country: 6986
    population    N  resistant  resistant %     years  missing BioProject %  year missing
UK (all, 2575) 2575        731         28.4 2001-2012                  42.4          1080
        Norway 3153        311          9.9 1999-2017                  99.9             0
    South Asia  238        159         66.8 2007-2021                  23.5            56

UK year missing among the 2,574 modelled genomes: 1080
South Asia by country (resistant/susceptible): {'Bangladesh': '1R/2S', 'India': '77R/38S', 'Pakistan': '81R/39S'}


## 2. Feature carriage (Table 3)

In [4]:
rows = []
for f in FEATURES + ['any of the eight']:
    r = {'feature': f}
    for name, df in pops.items():
        for lab, yy in [('R', 1), ('S', 0)]:
            s = df[df['y'] == yy]
            v = (s[FEATURES].sum(axis=1) > 0) if f == 'any of the eight' else s[f]
            r[f'{name} {lab} (n={len(s)})'] = f'{int(v.sum())} ({100 * v.mean():.1f}%)'
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

         feature UK R (n=730) UK S (n=1844) Norway R (n=311) Norway S (n=2842) South Asia R (n=159) South Asia S (n=79)
     gyrA_fq_mut  723 (99.0%)    164 (8.9%)      296 (95.2%)        138 (4.9%)          155 (97.5%)          36 (45.6%)
     parC_fq_mut  719 (98.5%)     18 (1.0%)      274 (88.1%)         56 (2.0%)          150 (94.3%)            1 (1.3%)
            qnrA     0 (0.0%)      1 (0.1%)         0 (0.0%)          0 (0.0%)             2 (1.3%)            0 (0.0%)
            qnrB     1 (0.1%)      3 (0.2%)         1 (0.3%)          2 (0.1%)             0 (0.0%)            0 (0.0%)
            qnrS     0 (0.0%)     28 (1.5%)         5 (1.6%)         10 (0.4%)           28 (17.6%)            6 (7.6%)
            oqxA     0 (0.0%)      0 (0.0%)         1 (0.3%)          0 (0.0%)             0 (0.0%)            0 (0.0%)
            oqxB     0 (0.0%)      0 (0.0%)         1 (0.3%)          0 (0.0%)             0 (0.0%)            0 (0.0%)
      aac6_Ib_cr  187 (25.6%)      2 (0.

## 3. Model, cross-validation, discrimination and calibration (Tables 4-5)

Eight-feature elastic-net logistic regression trained on the UK population. UK performance comes from five-fold
cross-validation in which each MLST sequence type stays within one fold. Norway and South Asia use the model fitted on all
UK genomes, without refitting. Point estimates are computed on the full evaluation set; intervals are percentile intervals
from 1,000 bootstrap resamples of genomes (seed 0).

In [5]:
X_uk, y_uk, groups_uk = uk_model[FEATURES].values, uk_model['y'].values, uk_model['MLST'].values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# no sequence type may appear on both sides of any fold
for k, (tr, te) in enumerate(sgkf.split(X_uk, y_uk, groups_uk)):
    assert not (set(groups_uk[tr]) & set(groups_uk[te])), f'MLST leakage in fold {k}'
print('leakage check passed: no sequence type is shared between training and validation parts of any fold')

def new_model():
    return LR(penalty='elasticnet', l1_ratio=0.5, solver='saga', C=1.0, max_iter=5000, random_state=SEED)

oof = cross_val_predict(new_model(), X_uk, y_uk, cv=sgkf, groups=groups_uk, method='predict_proba')[:, 1]
clf = new_model().fit(X_uk, y_uk)
print('\nfitted coefficients:', dict(zip(FEATURES, clf.coef_[0].round(3))), '| intercept', round(clf.intercept_[0], 3))
print('score of a genome with a parC mutation only:', round(clf.predict_proba(
    pd.DataFrame([{f: int(f == 'parC_fq_mut') for f in FEATURES}]).values)[0, 1], 3),
    '| parC + gyrA:', round(clf.predict_proba(pd.DataFrame([{f: int(f in ('parC_fq_mut', 'gyrA_fq_mut')) for f in FEATURES}]).values)[0, 1], 3))

p_ext = {'Norway': clf.predict_proba(norway[FEATURES].values)[:, 1], 'South Asia': clf.predict_proba(sasia[FEATURES].values)[:, 1]}
y_ext = {'Norway': norway['y'].values, 'South Asia': sasia['y'].values}

leakage check passed: no sequence type is shared between training and validation parts of any fold



fitted coefficients: {'gyrA_fq_mut': np.float64(3.741), 'parC_fq_mut': np.float64(6.266), 'qnrA': np.float64(0.0), 'qnrB': np.float64(0.712), 'qnrS': np.float64(0.0), 'oqxA': np.float64(0.0), 'oqxB': np.float64(0.0), 'aac6_Ib_cr': np.float64(0.871)} | intercept -6.315
score of a genome with a parC mutation only: 0.488 | parC + gyrA: 0.976


In [6]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def slope_int(y, p):
    m = LR(C=1e6, max_iter=1000).fit(logit(p).reshape(-1, 1), y)
    return m.coef_[0][0], m.intercept_[0]

def disc(y, p):
    t = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, t, labels=[0, 1]).ravel()
    return np.array([roc_auc_score(y, p), average_precision_score(y, p), tp / (tp + fn), tn / (tn + fp), matthews_corrcoef(y, t)])

def ece(y, p):
    idx = np.clip(np.digitize(p, np.linspace(0, 1, 11)) - 1, 0, 9)
    return sum((idx == b).sum() / len(p) * abs(p[idx == b].mean() - y[idx == b].mean()) for b in range(10) if (idx == b).sum() > 0)

def calib(y, p):
    s, i = slope_int(y, p)
    return np.array([s, i, brier_score_loss(y, np.clip(p, 1e-6, 1 - 1e-6)), ece(y, p)])

def boot(y, p, fn, n=1000):
    rng = np.random.default_rng(0)
    out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        out.append(fn(y[i], p[i]))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)

rule = lambda df: (df[FEATURES].sum(axis=1) > 0).astype(float).values     # rule-based baseline: any feature present
evalsets = {'UK (in-domain, 5-fold CV)': (y_uk, oof, rule(uk_model)),
            'Norway (external)': (y_ext['Norway'], p_ext['Norway'], rule(norway)),
            'South Asia (external)': (y_ext['South Asia'], p_ext['South Asia'], rule(sasia))}

print('TABLE 4  discrimination (point estimate and 95% bootstrap interval)')
for pop, (y, p, rb) in evalsets.items():
    for label, pp in [('elastic-net', p), ('rule-based', rb)]:
        est, (lo, hi) = disc(y, pp), boot(y, pp, disc)
        print(f'{pop:26s} {label:11s} ' + '  '.join(f'{n} {e:.3f} ({l:.3f}-{h:.3f})' for n, e, l, h in zip(['AUROC', 'AUPRC', 'Sens', 'Spec', 'MCC'], est, lo, hi)))

print('\nTABLE 5  calibration (elastic-net)')
for pop, (y, p, rb) in evalsets.items():
    est, (lo, hi) = calib(y, p), boot(y, p, calib)
    print(f'{pop:26s} N={len(y):5d}  ' + '  '.join(f'{n} {e:.4f} ({l:.4f}-{h:.4f})' for n, e, l, h in zip(['slope', 'intercept', 'Brier', 'ECE'], est, lo, hi)))

TABLE 4  discrimination (point estimate and 95% bootstrap interval)


UK (in-domain, 5-fold CV)  elastic-net AUROC 0.992 (0.986-0.996)  AUPRC 0.987 (0.979-0.994)  Sens 0.985 (0.976-0.993)  Spec 0.991 (0.987-0.995)  MCC 0.973 (0.964-0.983)


UK (in-domain, 5-fold CV)  rule-based  AUROC 0.938 (0.930-0.946)  AUPRC 0.770 (0.745-0.795)  Sens 0.990 (0.983-0.996)  Spec 0.886 (0.872-0.899)  MCC 0.822 (0.803-0.842)


Norway (external)          elastic-net AUROC 0.968 (0.954-0.981)  AUPRC 0.868 (0.830-0.902)  Sens 0.881 (0.843-0.916)  Spec 0.992 (0.988-0.995)  MCC 0.889 (0.860-0.917)


Norway (external)          rule-based  AUROC 0.945 (0.931-0.957)  AUPRC 0.594 (0.550-0.638)  Sens 0.955 (0.928-0.977)  Spec 0.935 (0.927-0.944)  MCC 0.738 (0.706-0.769)


South Asia (external)      elastic-net AUROC 0.970 (0.945-0.989)  AUPRC 0.981 (0.965-0.993)  Sens 0.943 (0.902-0.975)  Spec 0.987 (0.959-1.000)  MCC 0.910 (0.849-0.957)


South Asia (external)      rule-based  AUROC 0.731 (0.676-0.785)  AUPRC 0.789 (0.733-0.844)  Sens 0.994 (0.980-1.000)  Spec 0.468 (0.360-0.579)  MCC 0.594 (0.494-0.682)

TABLE 5  calibration (elastic-net)


UK (in-domain, 5-fold CV)  N= 2574  slope 1.0918 (0.9771-1.3063)  intercept -0.0414 (-0.6537-0.3063)  Brier 0.0088 (0.0058-0.0119)  ECE 0.0101 (0.0073-0.0137)


Norway (external)          N= 3153  slope 0.7259 (0.6768-0.7964)  intercept -0.4876 (-0.7718--0.2368)  Brier 0.0206 (0.0162-0.0251)  ECE 0.0176 (0.0126-0.0225)


South Asia (external)      N=  238  slope 0.7856 (0.6108-1.1409)  intercept 0.9273 (0.4478-1.4880)  Brier 0.0399 (0.0196-0.0670)  ECE 0.0347 (0.0128-0.0616)


## 4. Reliability: where the probabilities go wrong (Section 3.3, Supplementary Table S2)

In [7]:
def reliability(y, p):
    t = pd.DataFrame({'p': np.round(p, 4), 'y': y}).groupby('p')['y'].agg(['size', 'sum', 'mean'])
    t.columns = ['n', 'resistant', 'observed']
    return t.reset_index()

for pop in ['Norway', 'South Asia']:
    t = reliability(y_ext[pop], p_ext[pop])
    low, high = t[t['p'] < 0.1], t[t['p'] > 0.9]
    big = high.loc[high['n'].idxmax()]
    print(f"{pop}: predicted < 0.1: n={int(low['n'].sum())}, mean predicted={(low['p'] * low['n']).sum() / low['n'].sum():.3f}, "
          f"observed={low['resistant'].sum() / low['n'].sum():.3f} ({int(low['resistant'].sum())} resistant)")
    print(f"        predicted > 0.9: largest group n={int(big['n'])}, predicted={big['p']:.3f}, observed={big['observed']:.3f}")
    print(f"        mean predicted {p_ext[pop].mean():.3f} vs observed prevalence {y_ext[pop].mean():.3f}")
    print(t.to_string(index=False), '\n')

Norway: predicted < 0.1: n=2822, mean predicted=0.005, observed=0.013 (36 resistant)
        predicted > 0.9: largest group n=243, predicted=0.976, observed=0.909
        mean predicted 0.102 vs observed prevalence 0.099
     p    n  resistant  observed
0.0018 2683         15  0.005591
0.0037    1          0  0.000000
0.0043    2          0  0.000000
0.0087    1          0  0.000000
0.0708  135         21  0.155556
0.1541    1          1  1.000000
0.4878   32          0  0.000000
0.9757  243        221  0.909465
0.9879    1          1  1.000000
0.9897   54         52  0.962963 

South Asia: predicted < 0.1: n=87, mean predicted=0.034, observed=0.103 (9 resistant)
        predicted > 0.9: largest group n=116, predicted=0.976, observed=0.991
        mean predicted 0.633 vs observed prevalence 0.668
     p   n  resistant  observed
0.0018  47          4  0.085106
0.0708  40          5  0.125000
0.9757 116        115  0.991379
0.9897  35         35  1.000000 



## 5. Resistance mechanisms and lineages (Table 6, Sections 3.4 and 3.6)

In [8]:
ACQUIRED = ['qnrA', 'qnrB', 'qnrS', 'oqxA', 'oqxB', 'aac6_Ib_cr']
def mechanism(r):
    if r['parC_fq_mut'] == 1: return 'parC mutation'
    if any(r[g] == 1 for g in ACQUIRED): return 'acquired gene, no parC'
    return 'neither'

rows = []
for name, df in pops.items():
    r = df[df['y'] == 1].copy()
    r['mech'] = r.apply(mechanism, axis=1)
    r['P'] = clf.predict_proba(r[FEATURES].values)[:, 1]
    g = r.groupby('mech')['P'].agg(['size', 'mean'])
    for m_, row in g.iterrows():
        rows.append({'population': name, 'mechanism': m_, 'n': int(row['size']),
                     '% of resistant': round(100 * row['size'] / len(r), 1), 'mean predicted P': round(row['mean'], 3)})
print('TABLE 6'); print(pd.DataFrame(rows).to_string(index=False))

rn = norway[norway['y'] == 1].copy(); rn['mech'] = rn.apply(mechanism, axis=1)
nei = rn[rn['mech'] == 'neither']
print(f"\nNorway 'neither' genomes: {len(nei)}; distinct MLST types: {nei['MLST'].nunique()}; "
      f"ST131 share {100 * (nei['MLST'] == ST131).mean():.0f}%; gyrA mutation only: {int((nei['gyrA_fq_mut'] == 1).sum())}; "
      f"none of the eight features: {int((nei[FEATURES].sum(axis=1) == 0).sum())}; "
      f"collected outside 2007-2012 (known year): {int(((nei['Collection Year'] < 2007) | (nei['Collection Year'] > 2012)).sum())}")
print('resistant Norwegian genomes below the 0.5 threshold:', int((clf.predict_proba(rn[FEATURES].values)[:, 1] < 0.5).sum()))

print('\nLINEAGES (Section 3.6)')
for name, df in pops.items():
    s = df[df['MLST'] == ST131]
    print(f"{name:11s} MLST types {df['MLST'].nunique():3d} | ST131 {100 * len(s) / len(df):.1f}% ({len(s)}/{len(df)}) "
          f"| ST131 resistant {100 * s['y'].mean():.1f}% | top-5 share {100 * df['MLST'].value_counts().head(5).sum() / len(df):.1f}%")

TABLE 6
population              mechanism   n  % of resistant  mean predicted P
        UK acquired gene, no parC   1             0.1             0.134
        UK                neither  10             1.4             0.023
        UK          parC mutation 719            98.5             0.979
    Norway acquired gene, no parC   4             1.3             0.074
    Norway                neither  33            10.6             0.042
    Norway          parC mutation 274            88.1             0.978
South Asia acquired gene, no parC   7             4.4             0.041
South Asia                neither   2             1.3             0.036
South Asia          parC mutation 150            94.3             0.979

Norway 'neither' genomes: 33; distinct MLST types: 17; ST131 share 24%; gyrA mutation only: 19; none of the eight features: 14; collected outside 2007-2012 (known year): 31
resistant Norwegian genomes below the 0.5 threshold: 37

LINEAGES (Section 3.6)
UK          MLST t

## 6. Local recalibration experiment (Table 7)

Platt scaling (slope and intercept re-estimated on N randomly drawn target genomes, applied to the rest), 300 valid
repetitions per sample size, median and interquartile range of the calibration intercept on the held-out genomes. Seed for
each repetition: 1000 x N + attempt index; repetitions with a single phenotype class in either set are discarded and redrawn.

In [9]:
def recal_main(pop, Ns=(25, 50, 100, 150, 200), reps=300):
    y = pop['y'].values
    p = np.clip(clf.predict_proba(pop[FEATURES].values)[:, 1], 1e-6, 1 - 1e-6)
    lg = np.log(p / (1 - p))
    rows, disc_n = [], {}
    for N in Ns:
        valid = attempt = discards = 0
        while valid < reps and attempt < reps + 100:
            rng = np.random.default_rng(1000 * N + attempt)
            attempt += 1
            mask = np.zeros(len(y), bool); mask[rng.choice(len(y), N, replace=False)] = True
            if len(np.unique(y[mask])) < 2 or len(np.unique(y[~mask])) < 2:
                discards += 1; continue
            m = LR(C=1e6, max_iter=1000).fit(lg[mask].reshape(-1, 1), y[mask])
            ph = np.clip(1 / (1 + np.exp(-(m.coef_[0][0] * lg[~mask] + m.intercept_[0]))), 1e-6, 1 - 1e-6)
            i_after = LR(C=1e6, max_iter=1000).fit(np.log(ph / (1 - ph)).reshape(-1, 1), y[~mask]).intercept_[0]
            rows.append({'N': N, 'intercept': i_after, 'brier': brier_score_loss(y[~mask], ph)})
            valid += 1
        disc_n[N] = (valid, discards)
    return pd.DataFrame(rows), disc_n

print('TABLE 7')
for name, pop in [('Norway', norway), ('South Asia', sasia)]:
    df, dn = recal_main(pop)
    b_int, b_brier = slope_int(pop['y'].values, clf.predict_proba(pop[FEATURES].values)[:, 1])[1], brier_score_loss(pop['y'].values, clf.predict_proba(pop[FEATURES].values)[:, 1])
    print(f'\n{name}: before recalibration intercept {b_int:.3f}, Brier {b_brier:.4f}')
    for N in (25, 50, 100, 150, 200):
        s = df[df['N'] == N]
        q1, q3 = np.percentile(s['intercept'], [25, 75])
        print(f"  N={N:3d} median intercept {s['intercept'].median():7.3f} [{q1:7.3f}, {q3:7.3f}] | |intercept|>5: {int((s['intercept'].abs() > 5).sum()):2d}/300 "
              f"| median Brier {s['brier'].median():.4f} | valid {dn[N][0]}, discarded {dn[N][1]}")

TABLE 7



Norway: before recalibration intercept -0.488, Brier 0.0206
  N= 25 median intercept  -0.596 [ -0.870,  -0.379] | |intercept|>5:  9/300 | median Brier 0.0220 | valid 300, discarded 34
  N= 50 median intercept  -0.385 [ -0.650,   0.580] | |intercept|>5:  0/300 | median Brier 0.0208 | valid 300, discarded 2
  N=100 median intercept  -0.184 [ -0.629,   0.645] | |intercept|>5:  0/300 | median Brier 0.0200 | valid 300, discarded 0
  N=150 median intercept  -0.036 [ -0.497,   0.460] | |intercept|>5:  0/300 | median Brier 0.0196 | valid 300, discarded 0
  N=200 median intercept  -0.034 [ -0.389,   0.447] | |intercept|>5:  0/300 | median Brier 0.0196 | valid 300, discarded 0



South Asia: before recalibration intercept 0.927, Brier 0.0399
  N= 25 median intercept  -0.412 [ -0.909,   1.121] | |intercept|>5:  8/300 | median Brier 0.0469 | valid 300, discarded 0
  N= 50 median intercept  -0.192 [ -0.576,   0.384] | |intercept|>5:  5/300 | median Brier 0.0452 | valid 300, discarded 0
  N=100 median intercept  -0.077 [ -0.431,   0.338] | |intercept|>5:  1/300 | median Brier 0.0430 | valid 300, discarded 0
  N=150 median intercept   0.039 [ -0.369,   0.477] | |intercept|>5:  1/300 | median Brier 0.0429 | valid 300, discarded 0
  N=200 median intercept   0.249 [ -3.265,   0.907] | |intercept|>5: 17/300 | median Brier 0.0368 | valid 300, discarded 0


In [10]:
# Post hoc sensitivity analysis: the same splits fitted without a penalty (C = 1e6) and with a ridge penalty (C = 1).
# Splits: permute the target population with seed 1000 x repetition index + N and take the first N genomes.
def recal_ridge(pop, C, Ns=(25, 50, 100, 150, 200), reps=300):
    y = pop['y'].values
    p = clf.predict_proba(pop[FEATURES].values)[:, 1]
    lg = lambda q: logit(q).reshape(-1, 1)
    rows = []
    for N in Ns:
        vals = []
        for rep in range(reps):
            idx = np.random.default_rng(rep * 1000 + N).permutation(len(y)); ci, ei = idx[:N], idx[N:]
            yc, ye = y[ci], y[ei]
            if yc.sum() in (0, len(yc)) or ye.sum() in (0, len(ye)):
                continue
            m = LR(C=C, max_iter=1000).fit(lg(p[ci]), yc)
            pe = m.predict_proba(lg(p[ei]))[:, 1]
            vals.append(LR(C=C, max_iter=1000).fit(lg(pe), ye).intercept_[0])
        v = np.array(vals)
        rows.append({'N': N, 'reps': len(v), 'median': np.median(v), 'q25': np.percentile(v, 25), 'q75': np.percentile(v, 75),
                     '|int|>5': int((abs(v) > 5).sum())})
    return pd.DataFrame(rows).round(3)

for name, pop in [('Norway', norway), ('South Asia', sasia)]:
    for C in (1e6, 1.0):
        print(f'\n{name}, C={C:g}' + ('  (no penalty)' if C > 1 else '  (ridge)'))
        print(recal_ridge(pop, C).to_string(index=False))


Norway, C=1e+06  (no penalty)


  N  reps  median    q25    q75  |int|>5
 25   281  -0.596 -0.751 -0.395        2
 50   297  -0.336 -0.651  0.758        1
100   300  -0.196 -0.631  0.723        0
150   300  -0.006 -0.506  0.538        0
200   300   0.051 -0.395  0.445        0

Norway, C=1  (ridge)


  N  reps  median    q25   q75  |int|>5
 25   281  -0.120 -0.598 0.416        1
 50   297   0.191 -0.532 0.800        0
100   300   0.094 -0.480 0.682        0
150   300   0.071 -0.447 0.536        0
200   300   0.063 -0.380 0.444        0

South Asia, C=1e+06  (no penalty)


  N  reps  median    q25   q75  |int|>5
 25   300  -0.205 -0.800 1.111       16
 50   300  -0.138 -0.598 0.385       12
100   300  -0.067 -0.438 0.283        1
150   300   0.152 -0.345 0.576        0
200   300   0.233 -0.483 0.946       12

South Asia, C=1  (ridge)


  N  reps  median    q25   q75  |int|>5
 25   300   0.035 -0.616 0.821        0
 50   300  -0.004 -0.527 0.429        0
100   300  -0.054 -0.426 0.301        0
150   300   0.040 -0.380 0.445        0
200   300  -0.025 -0.568 0.417        0


## 7. Post hoc analyses (Tables 8-9)

In [11]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    unpen = LR(penalty=None, solver='lbfgs', max_iter=5000).fit(X_uk, y_uk)
print('TABLE 8  convergence warnings:', sorted({str(x.message) for x in w if issubclass(x.category, ConvergenceWarning)}) or 'none')
print(pd.DataFrame({'elastic_net': np.r_[clf.coef_[0], clf.intercept_[0]], 'unpenalized': np.r_[unpen.coef_[0], unpen.intercept_[0]]},
                   index=FEATURES + ['intercept']).round(3).to_string())

mcc = lambda y, p: matthews_corrcoef(y, (p >= 0.5).astype(int))
ci95 = lambda v: np.percentile(v, [2.5, 97.5])
mk = lambda: LR(penalty=None, solver='lbfgs', max_iter=5000)
X1 = uk_model[['parC_fq_mut']].values
p1_uk = cross_val_predict(mk(), X1, y_uk, cv=sgkf, groups=groups_uk, method='predict_proba')[:, 1]
m1 = mk().fit(X1, y_uk)
sets = {'UK (OOF)': (y_uk, p1_uk, oof),
        'Norway': (y_ext['Norway'], m1.predict_proba(norway[['parC_fq_mut']].values)[:, 1], p_ext['Norway']),
        'South Asia': (y_ext['South Asia'], m1.predict_proba(sasia[['parC_fq_mut']].values)[:, 1], p_ext['South Asia'])}
print('\nTABLE 9  parC-only versus eight-feature model (paired bootstrap of the differences)')
for name, (y, p1, p8) in sets.items():
    rng = np.random.default_rng(0); dA, dM = [], []
    for _ in range(1000):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        dA.append(roc_auc_score(y[i], p8[i]) - roc_auc_score(y[i], p1[i]))
        dM.append(mcc(y[i], p8[i]) - mcc(y[i], p1[i]))
    s1, i1 = slope_int(y, p1); s8, i8 = slope_int(y, p8)
    print(f'{name:11s} parC-only  AUROC={roc_auc_score(y, p1):.3f} MCC={mcc(y, p1):.3f} slope={s1:.3f} intercept={i1:.3f}')
    print(f'{"":11s} 8-feature  AUROC={roc_auc_score(y, p8):.3f} MCC={mcc(y, p8):.3f} slope={s8:.3f} intercept={i8:.3f}')
    print(f'{"":11s} difference AUROC={roc_auc_score(y, p8) - roc_auc_score(y, p1):+.3f} ({ci95(dA)[0]:+.3f} to {ci95(dA)[1]:+.3f})   '
          f'MCC={mcc(y, p8) - mcc(y, p1):+.3f} ({ci95(dM)[0]:+.3f} to {ci95(dM)[1]:+.3f})')

TABLE 8  convergence warnings: none
             elastic_net  unpenalized
gyrA_fq_mut        3.741        4.297
parC_fq_mut        6.266        6.901
qnrA               0.000       -0.012
qnrB               0.712        4.327
qnrS               0.000       -0.294
oqxA               0.000        0.000
oqxB               0.000        0.000
aac6_Ib_cr         0.871        1.214
intercept         -6.315       -7.178

TABLE 9  parC-only versus eight-feature model (paired bootstrap of the differences)


UK (OOF)    parC-only  AUROC=0.983 MCC=0.972 slope=1.000 intercept=0.145
            8-feature  AUROC=0.992 MCC=0.973 slope=1.092 intercept=-0.041
            difference AUROC=+0.008 (+0.004 to +0.013)   MCC=+0.001 (+0.000 to +0.003)


Norway      parC-only  AUROC=0.931 MCC=0.839 slope=0.671 intercept=-0.892
            8-feature  AUROC=0.968 MCC=0.889 slope=0.726 intercept=-0.488
            difference AUROC=+0.038 (+0.025 to +0.051)   MCC=+0.050 (+0.034 to +0.067)


South Asia  parC-only  AUROC=0.965 MCC=0.910 slope=0.812 intercept=1.993
            8-feature  AUROC=0.970 MCC=0.910 slope=0.786 intercept=0.927
            difference AUROC=+0.004 (-0.005 to +0.014)   MCC=+0.000 (+0.000 to +0.000)


## 8. Collection-period window, 2007-2012 (Table 10)

In [12]:
def window(df): return df[(df['Collection Year'] >= 2007) & (df['Collection Year'] <= 2012)]

def summarize(label, df):
    y = df['y'].values; p = clf.predict_proba(df[FEATURES].values)[:, 1]
    s, i = slope_int(y, p)
    print(f'{label:24s} n={len(y):5d} R={int(y.sum()):4d} prevalence={y.mean():.3f} meanP={p.mean():.3f} '
          f'AUROC={roc_auc_score(y, p):.3f} slope={s:.3f} intercept={i:.3f}')
    if len(y) >= 100:
        rng = np.random.default_rng(0); A, S, I = [], [], []
        for _ in range(1000):
            k = rng.integers(0, len(y), len(y))
            if len(np.unique(y[k])) < 2: continue
            A.append(roc_auc_score(y[k], p[k])); s_, i_ = slope_int(y[k], p[k]); S.append(s_); I.append(i_)
        print(f'{"":24s} 95% CI  AUROC {ci95(A)[0]:.3f}-{ci95(A)[1]:.3f}  slope {ci95(S)[0]:.3f}-{ci95(S)[1]:.3f}  intercept {ci95(I)[0]:.3f} to {ci95(I)[1]:.3f}')

uk_w = window(uk_model)
print(f"UK in 2007-2012: n={len(uk_w)}, resistant={int(uk_w['y'].sum())} ({100 * uk_w['y'].mean():.1f}%)")
print('TABLE 10')
for name, df in [('Norway', norway), ('South Asia', sasia)]:
    summarize(f'{name} full', df); summarize(f'{name} 2007-2012', window(df))

print('\nNorway, resistant genomes: mechanism composition (%) and ST131 share of all genomes')
for label, df in [('full', norway), ('2007-2012', window(norway))]:
    r_ = df[df['y'] == 1].copy(); r_['mech'] = r_.apply(mechanism, axis=1)
    print(f'  {label:10s}', (r_['mech'].value_counts(normalize=True) * 100).round(1).to_dict(), ' ST131 share:', round(100 * (df['MLST'] == ST131).mean(), 1))
sw = window(sasia)
print(f"\nSouth Asia 2007-2012: {len(sw)} genomes, {int(sw['y'].sum())} resistant; resistant genomes collected in 2007-2012: "
      f"{int(sw['y'].sum())} of {int(sasia['y'].sum())}; susceptible: {int((sw['y'] == 0).sum())} of {int((sasia['y'] == 0).sum())}")

UK in 2007-2012: n=884, resistant=180 (20.4%)
TABLE 10
Norway full              n= 3153 R= 311 prevalence=0.099 meanP=0.102 AUROC=0.968 slope=0.726 intercept=-0.488


                         95% CI  AUROC 0.954-0.981  slope 0.677-0.796  intercept -0.772 to -0.237
Norway 2007-2012         n= 1126 R=  88 prevalence=0.078 meanP=0.091 AUROC=0.986 slope=1.003 intercept=-1.464


                         95% CI  AUROC 0.967-0.999  slope 0.814-5.303  intercept -17.429 to -0.819
South Asia full          n=  238 R= 159 prevalence=0.668 meanP=0.633 AUROC=0.970 slope=0.786 intercept=0.927


                         95% CI  AUROC 0.945-0.989  slope 0.611-1.141  intercept 0.448 to 1.488
South Asia 2007-2012     n=   62 R=   8 prevalence=0.129 meanP=0.151 AUROC=1.000 slope=3.254 intercept=-0.963

Norway, resistant genomes: mechanism composition (%) and ST131 share of all genomes
  full       {'parC mutation': 88.1, 'neither': 10.6, 'acquired gene, no parC': 1.3}  ST131 share: 7.9
  2007-2012  {'parC mutation': 97.7, 'neither': 2.3}  ST131 share: 6.7

South Asia 2007-2012: 62 genomes, 8 resistant; resistant genomes collected in 2007-2012: 8 of 159; susceptible: 54 of 79


## 9. South Asia sensitivity check: sampling by BioProject

South Asia is a small pooled population assembled from a few submitting studies. One BioProject (PRJNA925003) contributes
35 genomes, all resistant. This section shows how much of the South Asian calibration result depends on that study.

In [13]:
print('South Asia by BioProject (genomes, resistant, share resistant, years):')
g = sasia.groupby('BioProject Accession', dropna=False)
print(pd.DataFrame({'n': g.size(), 'resistant': g['y'].sum(), 'share resistant': g['y'].mean().round(3),
                    'years': g['Collection Year'].agg(lambda s: f'{int(s.min())}-{int(s.max())}' if s.notna().any() else 'missing')}
                   ).sort_values('n', ascending=False).to_string())

def report(label, df):
    y = df['y'].values; p = clf.predict_proba(df[FEATURES].values)[:, 1]
    s, i = slope_int(y, p)
    print(f'{label:34s} n={len(df):4d} resistant={int(y.sum()):4d} prevalence={y.mean():.3f} meanP={p.mean():.3f} '
          f'AUROC={roc_auc_score(y, p):.3f} slope={s:.3f} intercept={i:.3f} Brier={brier_score_loss(y, p):.4f}')

print()
report('South Asia, all', sasia)
report('South Asia, without PRJNA925003', sasia[sasia['BioProject Accession'] != 'PRJNA925003'])

South Asia by BioProject (genomes, resistant, share resistant, years):
                       n  resistant  share resistant      years
BioProject Accession                                           
PRJNA522294           85         67            0.788  2015-2017
PRJEB2827             58          8            0.138  2007-2010
NaN                   56         49            0.875  2019-2021
PRJNA925003           35         35            1.000    missing
PRJEB2226              4          0            0.000  2008-2009

South Asia, all                    n= 238 resistant= 159 prevalence=0.668 meanP=0.633 AUROC=0.970 slope=0.786 intercept=0.927 Brier=0.0399
South Asia, without PRJNA925003    n= 203 resistant= 124 prevalence=0.611 meanP=0.588 AUROC=0.980 slope=0.954 intercept=0.765 Brier=0.0320


---
*Environment: Python 3.13 on Google Colab, scikit-learn 1.6.1, pandas 2.2.3, numpy 2.1.3. Figures 1-3 are in `results/`.*